<a href="https://colab.research.google.com/github/gangababupokanati/ai-mentor-portfolio/blob/main/Day9_StudyAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search

import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Gemini API key: ··········


In [3]:
!pip install -q -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 12.3 MB/s eta 0:00:00


In [4]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test it works
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

...Hiring 2026 is a flagship national-level recruitment program by Tata Consultancy Services (TCS) exclusively for science and computer application graduates from the 2025 and 2026... TCS MBA Hiring 2026: Tata Consultancy Services (TCS) has officially announced its MBA Hiring Drive for the Batch … Tata Consultancy Services (TCS), a global leader in IT services, consulting, and business solutions, 


In [5]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_1668/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [6]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': 'fe5cabe7-eadb-4883-ac3a-bd6fcd257fa8', 'type': 'tool_call'}]

[2] ToolMessage
    Content: India's largest IT services firm, Tata Consultancy Services (TCS), has made 25,000 job offers to fresh graduates for FY2026-27, signaling a cautious yet continued commitment to campus hiring ... TCS Hiring 2026: 40 अरब डॉलर की डील! अब देश की सबसे बड़ी IT कंपनी में फ्रेशर्स के लिए 25 हजार नई नौकरियां

[3] AIMessage
    Content: [{'type': 'text', 'text': 'TCS has made 25,000 job offers to fresh graduates for FY2026-27.', 'extras': {'signature': 'CrgCAQw51sc+nHzJKTIIZqlG7ONLlQjAIBx+aquDcEx3cOpxS76WeodK3nclD4SMmI53P8wdkntmv/1h7pk8LDGXkzKtkL+fAy/bC2T/pUXGJ5LT+qTxFJSuFY9NRKsVTxBjd0NRnlO8sfqc7Bc7eMEGN7aCEdOoABNBNOl4dXNGj/kw8ytpv


## Day 9 Lab 9A — Trace as a story

1. **Human asked:** "What is TCS's 2026 hiring quota?"
2. **Agent thought:** "I don't know recent figures. I should search."
3. **Agent acted:** called `web_search('TCS 2026 hiring quota')`.
4. **Agent observed:** got back search results mentioning 40-50K range.
5. **Agent answered:** synthesised "Based on search results, TCS plans to hire 40-50K freshers..."

This is the ReAct loop. Every agent we build follows this pattern.

In [9]:
result = agent.invoke({
    'messages': [('user', 'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')


[0] HumanMessage
    Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    [{'type': 'text', 'text': 'I cannot directly search a URL and tell you what it says. The `web_search` tool takes a search query, not a URL.', 'extras': {'signature': 'CosDAQw51seVVtnjQhKSqZCK6odwTFQwyG9WDf/4BsDzve7oGh0wnoZmOiE87l8ZMvpPxLHTjvE23hm+dIba6rZeO8Q+LLsxvdDCbJip8cTF4MzMLNcVwyoqhE/9xzDIbBelG


In [10]:
!pip install -q beautifulsoup4 requests

In [11]:
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool

@tool
def jd_fetcher(url: str) -> str:
    """Fetch a job description from a URL and return clean plain text.
    Use ONLY when the user provides a job posting URL and you need the JD content.
    Returns first 4000 characters of the cleaned page text."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:4000]
    except Exception as e:
        return f'ERROR: failed to fetch URL — {e}'

# Test it — should return ERROR (bad URL is intentional here)
print(jd_fetcher.invoke({'url': 'https://example.com'}))

Example Domain
Example Domain
This domain is for use in documentation examples without needing permission. Avoid use in operations.
Learn more


In [12]:
@tool
def skills_gap(student_skills: str, must_have_skills: str) -> str:
    """Compare a student's skills to a job's must-have skills (both comma-separated).
    Returns missing skills comma-separated, or 'none' if student has all.
    Use when the user provides a student profile and a JD's required skills."""
    a = set(s.strip().lower() for s in student_skills.split(',') if s.strip())
    b = set(s.strip().lower() for s in must_have_skills.split(',') if s.strip())
    missing = sorted(b - a)
    return ', '.join(missing) if missing else 'none'

# Test it
print(skills_gap.invoke({
    'student_skills': 'Python, Java, SQL',
    'must_have_skills': 'Python, Java, SQL, Spring Boot, AWS',
}))
# Should print: aws, spring boot

aws, spring boot


In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

@tool
def answer_scorer(question: str, answer: str) -> str:
    """Score a student's answer to a placement interview question, 1-10, with one-line rationale.
    Use when evaluating how well a student answered a specific interview question.
    Returns format: 'Score: X/10. Rationale: <reason>'"""
    prompt = (f'Score this placement interview answer 1-10 with one-line rationale.\n'
              f'Question: {question}\n'
              f'Answer: {answer}')
    return llm.invoke(prompt).content

# Test it
print(answer_scorer.invoke({
    'question': 'Why TCS Digital?',
    'answer': 'Because TCS is big and they pay well.',
}))
# Should give a low score like 3/10 with reason

2/10. It's generic, shows no research into the "Digital" aspect, and explicitly prioritizes compensation, which is a major red flag in an interview.


In [14]:
from langgraph.prebuilt import create_react_agent

tools = [jd_fetcher, skills_gap, answer_scorer]
agent = create_react_agent(llm, tools=tools)
print(f'Agent created with {len(tools)} tools.')

Agent created with 3 tools.


/tmp/ipykernel_1668/3943891205.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools)


In [15]:
import json, pathlib

# Create the data folder and profiles file
pathlib.Path('data').mkdir(exist_ok=True)

profiles = [
    {
        "name": "Ravi Kumar",
        "branch": "CSE",
        "cgpa": 8.2,
        "target_company": "TCS Digital",
        "skills": ["Python", "Java", "SQL", "Data Structures"]
    },
    {
        "name": "Sneha Reddy",
        "branch": "ECE",
        "cgpa": 7.8,
        "target_company": "Cognizant",
        "skills": ["Python", "C++", "Communication", "Problem Solving"]
    },
    {
        "name": "Arun Pillai",
        "branch": "IT",
        "cgpa": 8.9,
        "target_company": "Amazon",
        "skills": ["Python", "Java", "AWS", "SQL", "Spring Boot", "System Design"]
    }
]

pathlib.Path('data/student_profiles.json').write_text(json.dumps(profiles, indent=2))
print('Profiles saved!')

Profiles saved!


In [16]:
profiles = json.loads(pathlib.Path('data/student_profiles.json').read_text())

for i, p in enumerate(profiles):
    print(f'\n{"="*70}')
    print(f'Student {i+1}: {p["name"]} — {p["branch"]} CGPA {p["cgpa"]} → {p["target_company"]}')
    print(f'{"="*70}')

    msg = (f"I am {p['name']}, B.Tech {p['branch']} CGPA {p['cgpa']}, "
           f"skills: {', '.join(p['skills'])}. Target: {p['target_company']}. "
           f"Plan 3 mock interview questions for me, score one of my sample answers, "
           f"and tell me what skills I need to add to be a strong fit.")

    result = agent.invoke(
        {'messages': [('user', msg)]},
        config={'recursion_limit': 10}
    )

    for j, m in enumerate(result['messages']):
        print(f'\n  [{j}] {type(m).__name__}')
        if hasattr(m, 'content') and m.content:
            print(f'      {str(m.content)[:300]}')
        if hasattr(m, 'tool_calls') and m.tool_calls:
            for tc in m.tool_calls:
                print(f'      → tool_call: {tc.get("name")}({tc.get("args")})')


Student 1: Ravi Kumar — CSE CGPA 8.2 → TCS Digital

  [0] HumanMessage
      I am Ravi Kumar, B.Tech CSE CGPA 8.2, skills: Python, Java, SQL, Data Structures. Target: TCS Digital. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skills I need to add to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': 'Hello Ravi! That\'s a great profile. Let\'s get you ready for TCS Digital.\n\nHere are 3 mock interview questions for you, covering key areas:\n\n1.  **Data Structures & Algorithms:** "Explain the concept of a Hash Map (or Dictionary) and discuss its time complexity for co

Student 2: Sneha Reddy — ECE CGPA 7.8 → Cognizant

  [0] HumanMessage
      I am Sneha Reddy, B.Tech ECE CGPA 7.8, skills: Python, C++, Communication, Problem Solving. Target: Cognizant. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skills I need to add to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text'

In [17]:
result = agent.invoke({
    'messages': [('user', 'Fetch this JD and tell me the must-have skills: '
                          'https://this-does-not-exist-99999.example.com/jd')]
}, config={'recursion_limit': 5})

print('Failure recovery trace:')
for j, m in enumerate(result['messages']):
    print(f'\n[{j}] {type(m).__name__}')
    if hasattr(m, 'content') and m.content:
        print(f'    {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        for tc in m.tool_calls:
            print(f'    → {tc.get("name")}({tc.get("args")})')

Failure recovery trace:

[0] HumanMessage
    Fetch this JD and tell me the must-have skills: https://this-does-not-exist-99999.example.com/jd

[1] AIMessage
    → jd_fetcher({'url': 'https://this-does-not-exist-99999.example.com/jd'})

[2] ToolMessage
    ERROR: failed to fetch URL — HTTPSConnectionPool(host='this-does-not-exist-99999.example.com', port=443): Max retries exceeded with url: /jd (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x78cad4323680>: Failed to resolve 'this-does-not-exist-99999.example.com' ([Errn

[3] AIMessage
    [{'type': 'text', 'text': 'I was unable to fetch the job description from the URL provided. It seems to be an invalid link. Please provide a valid job posting URL so I can assist you further.', 'extras': {'signature': 'CqkDAQw51sfRT1PalmDFEcq1DaF7VyEU2vOfnK/blj3kY1RKA3mB7fEVZ/vsrCAdYZd5IPBxF1FMLqseg
